# Residual-gas background and Auger diagnostics

This cleaned notebook keeps the defensible analysis path from the original notebook:

- inspect the possible Auger structure at progressively earlier processing stages;
- compare the two wavelength-scan families without merging them;
- show GMD, electron counts, and counts/GMD for both families;
- construct GMD-normalised, Jacobian-corrected maps on the native unequal KE bins;
- subtract only residual-gas spectra measured at the same photon energy;
- repeat the maps from shot-resolved H5 counts as the primary cross-check.

Aggregate files are loaded with a lightweight loader that omits covariance matrices. Binding-energy maps, interpolation to equal KE bins, smoothing variants, merged scan families, and loose exploratory cells are intentionally omitted. In background-subtracted maps, rows without a measured background remain `NaN`; they are not silently replaced by unsubtracted signal.

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

%matplotlib inline

os.environ.setdefault("FLASH_ENV", "remote")


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "analysis" / "scripts").exists():
            return candidate
    fallback = Path("/home/mmanresa/glycine26")
    if (fallback / "analysis" / "scripts").exists():
        return fallback
    raise RuntimeError("Could not find repo root containing analysis/scripts")


repo_root = find_repo_root()
scripts_dir = repo_root / "analysis" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import config
from compute_aggregates import AggregatesData


def load_aggregates_etof(path):
    # Load eTOF means/bookkeeping while skipping covariance matrices.
    path = Path(path)
    with h5py.File(path, "r") as handle:
        metadata = {}
        for key in handle.attrs:
            value = handle.attrs[key]
            if isinstance(value, bytes):
                value = value.decode()
            elif isinstance(value, np.ndarray):
                value = value.tolist()
            metadata[key] = value

        def optional(key):
            return handle[key][:] if key in handle else None

        return AggregatesData(
            G=handle["G"][:],
            GtG=optional("GtG"),
            n_per_bin=handle["n_per_bin"][:],
            gmd_edges=handle["gmd_edges"][:],
            D=optional("D"),
            tof_edges=optional("tof_edges"),
            vls_pixels=optional("vls_pixels"),
            background=optional("background"),
            ion_tof_edges=optional("ion_tof_edges"),
            z_edges=optional("z_edges"),
            nominal_energies=optional("nominal_energies"),
            metadata=metadata,
        )


print("repo_root   :", repo_root)
print("FLASH_ENV   :", os.environ.get("FLASH_ENV"))
print("COMBINED_DIR:", config.COMBINED_DIR)

In [ ]:
# File selection
SIGNAL_AGG_DIR = Path(config.COMBINED_DIR)
SIGNAL_AGG_GLOB = "glycine_WL_scan_*eV_aggregates_etof1.h5"
BACKGROUND_AGG_DIR = SIGNAL_AGG_DIR / "background_run58890_per_energy"
BACKGROUND_AGG_GLOB = "background_*eV_run58890_aggregates_etof1.h5"
MIN_ENERGY = None
MAX_ENERGY = 288.0

# Main analysis choices
NORMALISE_BY_GMD = True
APPLY_JACOBIAN = True
BACKGROUND_SCALE = 1.0
CHUNK_TRAINS = 1000
TOF_RANGE = (800, 1100)
KE_XLIM = (180, 280)

# Upstream diagnostic run
UPSTREAM_DATASET = "floating"
UPSTREAM_ENERGY = 274.0
UPSTREAM_GMD_BIN = 0

# Map display. Change dataset and limits, then rerun the plotting cells.
MAP_DATASET = "floating"  # "floating" for .0/.5 files, or "integer"
SIGNAL_VMAX = None
BACKGROUND_VMAX = None
SUBTRACTED_ABS_VMAX = None
INVERT_PHOTON_AXIS = True

CALIB_PATH = (
    Path(config.COMBINED_DIR).parent
    / "Calibrations"
    / "etof_energy_calibration_t0_scan.json"
)
with CALIB_PATH.open("r", encoding="utf-8") as handle:
    calibration = json.load(handle)

best_t0 = float(calibration["t0_tof_units"])
best_slope = float(calibration["slope_eV_tof_unit2"])
best_E0 = float(calibration["E0_eV"])


def tof_to_ke(tof):
    tof = np.asarray(tof, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ke = best_slope / (tof - best_t0) ** 2 + best_E0
    return np.where(tof > best_t0, ke, np.nan)


print("Calibration:", CALIB_PATH)
print(f"t0={best_t0:.6f}, slope={best_slope:.6e}, E0={best_E0:.6f} eV")

In [ ]:
SIGNAL_ENERGY_RE = re.compile(
    r"^glycine_WL_scan_(\d+(?:\.\d+)?)eV_aggregates_etof1$"
)
BACKGROUND_ENERGY_RE = re.compile(
    r"^background_(\d+(?:\.\d+)?)eV_run58890_aggregates_etof1$"
)


def energy_from_path(path, pattern):
    match = pattern.search(Path(path).stem)
    if match is None:
        raise ValueError(f"Could not parse photon energy from {Path(path).name}")
    return float(match.group(1))


def scan_family_from_path(path):
    match = SIGNAL_ENERGY_RE.search(Path(path).stem)
    if match is None:
        return "unknown"
    return "floating" if "." in match.group(1) else "integer"


def half_step_edges(values):
    values = np.asarray(values, dtype=float)
    if values.size == 1:
        return np.array([values[0] - 0.5, values[0] + 0.5])
    if np.any(np.diff(values) <= 0):
        raise ValueError("Map energies must be strictly increasing within one scan family")
    midpoints = 0.5 * (values[:-1] + values[1:])
    return np.concatenate(
        [[2 * values[0] - midpoints[0]], midpoints, [2 * values[-1] - midpoints[-1]]]
    )


def robust_upper(data, percentile=99.5, default=1.0):
    finite = np.asarray(data, dtype=float)
    finite = finite[np.isfinite(finite)]
    return float(np.nanpercentile(finite, percentile)) if finite.size else default


def robust_abs(data, percentile=99.0, default=1.0):
    finite = np.asarray(data, dtype=float)
    finite = np.abs(finite[np.isfinite(finite)])
    value = float(np.nanpercentile(finite, percentile)) if finite.size else default
    return value if np.isfinite(value) and value > 0 else default


def keep_mask_like_compute_aggregates(handle, trim_start=0, trim_end=0):
    if "between_tdc_files" not in handle:
        return np.ones(handle["gmd"].shape[0], dtype=bool)
    between_files = handle["between_tdc_files"][:].astype(bool)
    good_positions = np.where(~between_files)[0]
    if trim_end > 0:
        good_positions = good_positions[trim_start:-trim_end]
    else:
        good_positions = good_positions[trim_start:]
    keep = np.zeros(between_files.shape[0], dtype=bool)
    keep[good_positions] = True
    return keep


def resolve_source_h5(aggregate_path, metadata):
    raw = metadata.get("input_h5", "")
    if isinstance(raw, bytes):
        raw = raw.decode()
    raw = Path(raw)
    candidates = [
        raw,
        Path(aggregate_path).parent / raw.name,
        Path(config.COMBINED_DIR) / raw.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return raw


def aggregate_spectrum(agg, normalise_by_gmd=NORMALISE_BY_GMD):
    n = np.asarray(agg.n_per_bin, dtype=float)
    counts = np.nansum(np.nan_to_num(agg.D, nan=0.0) * n[:, None], axis=0)
    sum_gmd = float(np.nansum(np.nan_to_num(agg.G, nan=0.0) * n))
    shots = int(np.nansum(n))
    denominator = sum_gmd if normalise_by_gmd else shots
    spectrum = counts / denominator if denominator > 0 else np.full_like(counts, np.nan)
    return {
        "spectrum": spectrum,
        "counts": counts,
        "sum_gmd": sum_gmd,
        "shots": shots,
        "hits": float(np.nansum(counts)),
    }


def shot_resolved_spectrum(
    source_h5,
    tof_edges,
    gmd_edges,
    *,
    trim_start=0,
    trim_end=0,
    normalise_by_gmd=NORMALISE_BY_GMD,
    chunk_trains=CHUNK_TRAINS,
):
    counts = np.zeros(len(tof_edges) - 1, dtype=float)
    sum_gmd = 0.0
    shots = 0
    saturated_shots = 0
    max_slots = 0
    gmd_edges = np.asarray(gmd_edges, dtype=float)

    with h5py.File(source_h5, "r") as handle:
        keep = keep_mask_like_compute_aggregates(handle, trim_start, trim_end)
        n_trains = handle["gmd"].shape[0]
        max_slots = int(handle["tofs_e"].shape[-1])
        for start in range(0, n_trains, chunk_trains):
            stop = min(start + chunk_trains, n_trains)
            row_keep = keep[start:stop]
            if not row_keep.any():
                continue
            gmd = handle["gmd"][start:stop][row_keep].ravel()
            tofs = handle["tofs_e"][start:stop][row_keep]
            tofs = tofs.reshape(-1, tofs.shape[-1])
            gmd_bin = np.digitize(gmd, gmd_edges) - 1
            selected = (
                np.isfinite(gmd)
                & (gmd_bin >= 0)
                & (gmd_bin < len(gmd_edges) - 1)
            )
            if not selected.any():
                continue
            selected_tofs = tofs[selected]
            hits = selected_tofs.ravel()
            hits = hits[hits > 0]
            hist, _ = np.histogram(hits, bins=tof_edges)
            counts += hist
            sum_gmd += float(np.sum(gmd[selected]))
            shots += int(np.count_nonzero(selected))
            saturated_shots += int(
                np.count_nonzero(np.count_nonzero(selected_tofs > 0, axis=1) >= max_slots)
            )

    denominator = sum_gmd if normalise_by_gmd else shots
    spectrum = counts / denominator if denominator > 0 else np.full_like(counts, np.nan)
    return {
        "spectrum": spectrum,
        "counts": counts,
        "sum_gmd": sum_gmd,
        "shots": shots,
        "hits": float(np.sum(counts)),
        "saturated_shots": saturated_shots,
        "max_slots": max_slots,
    }


def tof_window_to_ke_edges_and_data(
    tof_edges, data, tof_range=TOF_RANGE, apply_jacobian=APPLY_JACOBIAN
):
    tof_edges = np.asarray(tof_edges, dtype=float)
    data = np.asarray(data, dtype=float)
    tof_centres = 0.5 * (tof_edges[:-1] + tof_edges[1:])
    mask = np.ones(tof_centres.shape, dtype=bool)
    if tof_range[0] is not None:
        mask &= tof_centres >= tof_range[0]
    if tof_range[1] is not None:
        mask &= tof_centres <= tof_range[1]
    indices = np.where(mask)[0]
    if not indices.size:
        raise ValueError(f"No TOF bins inside {tof_range}")
    lo, hi = int(indices[0]), int(indices[-1]) + 1
    ke_edges = tof_to_ke(tof_edges[lo : hi + 1])
    if not np.all(np.isfinite(ke_edges)):
        raise ValueError("Selected TOF range crosses the calibration singularity")
    converted = data[..., lo:hi].astype(float)
    if apply_jacobian:
        converted = converted / np.abs(np.diff(ke_edges))
    if ke_edges[0] > ke_edges[-1]:
        ke_edges = ke_edges[::-1]
        converted = converted[..., ::-1]
    return ke_edges, converted


def spectrum_units():
    units = "counts / uJ" if NORMALISE_BY_GMD else "counts / shot"
    return f"{units} / eV" if APPLY_JACOBIAN else f"{units} / native TOF bin"

In [ ]:
signal_records = []
for path in sorted(SIGNAL_AGG_DIR.glob(SIGNAL_AGG_GLOB)):
    energy = energy_from_path(path, SIGNAL_ENERGY_RE)
    if MIN_ENERGY is not None and energy < MIN_ENERGY:
        continue
    if MAX_ENERGY is not None and energy > MAX_ENERGY:
        continue
    agg = load_aggregates_etof(path)
    if agg.config != 1 or agg.mode != "spectral" or agg.D is None:
        print(f"Skipping incompatible aggregate: {path.name}")
        continue
    signal_records.append(
        {
            "path": path,
            "energy": energy,
            "family": scan_family_from_path(path),
            "agg": agg,
            "source_h5": resolve_source_h5(path, agg.metadata),
        }
    )

background_records = []
for path in sorted(BACKGROUND_AGG_DIR.glob(BACKGROUND_AGG_GLOB)):
    energy = energy_from_path(path, BACKGROUND_ENERGY_RE)
    agg = load_aggregates_etof(path)
    if agg.config != 1 or agg.mode != "spectral" or agg.D is None:
        print(f"Skipping incompatible background aggregate: {path.name}")
        continue
    background_records.append(
        {
            "path": path,
            "energy": energy,
            "agg": agg,
            "source_h5": resolve_source_h5(path, agg.metadata),
        }
    )

signal_records.sort(key=lambda row: (row["family"], row["energy"], row["path"].name))
background_records.sort(key=lambda row: (row["energy"], row["path"].name))

if not signal_records:
    raise FileNotFoundError(f"No signal aggregates matched {SIGNAL_AGG_DIR / SIGNAL_AGG_GLOB}")

print(f"Signal aggregates: {len(signal_records)}")
for family in ("floating", "integer"):
    family_rows = [row for row in signal_records if row["family"] == family]
    print(f"  {family:8s}: {len(family_rows)} files")
    for row in family_rows:
        print(f"    {row['energy']:8.3f} eV  {row['path'].name}")
print(f"Background aggregates: {len(background_records)}")
print("Background energies:", [row["energy"] for row in background_records])

## Scan-level intensity diagnostics

These totals come from the lightweight aggregate arrays. Since `D`, `G`, and `n_per_bin` are per-bin means and shot counts, total counts are reconstructed as `sum(D * n)` and summed GMD as `sum(G * n)`. The two scan families are shown separately and are never merged.

In [ ]:
diagnostic_rows = []
for record in signal_records:
    summary = aggregate_spectrum(record["agg"])
    shots = summary["shots"]
    sum_gmd = summary["sum_gmd"]
    counts = summary["hits"]
    diagnostic_rows.append(
        {
            "energy": record["energy"],
            "family": record["family"],
            "name": record["path"].name,
            "shots": shots,
            "gmd_per_shot": sum_gmd / shots if shots else np.nan,
            "counts_per_shot": counts / shots if shots else np.nan,
            "counts_per_gmd": counts / sum_gmd if sum_gmd > 0 else np.nan,
        }
    )

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True, constrained_layout=True)
styles = {"floating": ("o", ".0/.5 scan"), "integer": ("s", "integer scan")}
for family, (marker, label) in styles.items():
    rows = sorted(
        [row for row in diagnostic_rows if row["family"] == family],
        key=lambda row: row["energy"],
    )
    if not rows:
        continue
    energy = np.array([row["energy"] for row in rows])
    axes[0].plot(
        energy,
        [row["gmd_per_shot"] for row in rows],
        marker + "-",
        label=label,
    )
    axes[1].plot(
        energy,
        [row["counts_per_shot"] for row in rows],
        marker + "-",
        label=label,
    )
    axes[2].plot(
        energy,
        [row["counts_per_gmd"] for row in rows],
        marker + "-",
        label=label,
    )

axes[0].set_ylabel("summed GMD / shots (uJ)")
axes[1].set_ylabel("eTOF counts / shot")
axes[2].set_ylabel("eTOF counts / summed GMD")
axes[2].set_xlabel("incident photon energy (eV)")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()

print("Per-file diagnostics:")
for row in sorted(diagnostic_rows, key=lambda row: (row["family"], row["energy"])):
    print(
        f"{row['energy']:8.3f} eV  {row['family']:8s}  "
        f"shots={row['shots']:8d}  GMD/shot={row['gmd_per_shot']:.5g}  "
        f"counts/shot={row['counts_per_shot']:.5g}  "
        f"counts/GMD={row['counts_per_gmd']:.5g}  {row['name']}"
    )

## Upstream Auger checks

Choose one run above. The following cells move upstream one level at a time while keeping the same run, train mask, GMD acceptance, and TOF bins.

In [ ]:
upstream_candidates = [
    row for row in signal_records if row["family"] == UPSTREAM_DATASET
]
if not upstream_candidates:
    raise ValueError(f"No signal files in dataset {UPSTREAM_DATASET!r}")
upstream_record = min(
    upstream_candidates, key=lambda row: abs(row["energy"] - UPSTREAM_ENERGY)
)
upstream_agg = upstream_record["agg"]
upstream_source_h5 = upstream_record["source_h5"]
upstream_tof_edges = np.asarray(upstream_agg.tof_edges, dtype=float)
upstream_tof_centres = 0.5 * (upstream_tof_edges[:-1] + upstream_tof_edges[1:])
upstream_trim_start = int(upstream_agg.metadata.get("trim_start", 0))
upstream_trim_end = int(upstream_agg.metadata.get("trim_end", 0))

print(
    f"Requested {UPSTREAM_ENERGY:.3f} eV in {UPSTREAM_DATASET!r}; "
    f"using {upstream_record['energy']:.3f} eV"
)
print("Aggregate:", upstream_record["path"])
print("Source H5:", upstream_source_h5)
print(f"Trim start/end: {upstream_trim_start}, {upstream_trim_end}")
print("GMD edges:", upstream_agg.gmd_edges)

In [ ]:
# Level 1: final GMD-normalised, calibrated spectrum for each aggregate GMD bin.
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
cmap = plt.get_cmap("viridis")
for g in range(upstream_agg.n_gmd_bins):
    n = int(upstream_agg.n_per_bin[g])
    sum_gmd = float(upstream_agg.G[g] * n)
    counts = np.nan_to_num(upstream_agg.D[g], nan=0.0) * n
    denominator = sum_gmd if NORMALISE_BY_GMD else n
    spectrum = counts / denominator if denominator > 0 else np.full_like(counts, np.nan)
    ke_edges, y = tof_window_to_ke_edges_and_data(upstream_tof_edges, spectrum)
    ke_centres = 0.5 * (ke_edges[:-1] + ke_edges[1:])
    label = (
        f"[{upstream_agg.gmd_edges[g]:.2g}, "
        f"{upstream_agg.gmd_edges[g + 1]:.2g}) uJ"
    )
    ax.plot(ke_centres, y, lw=0.9, color=cmap(g / max(upstream_agg.n_gmd_bins - 1, 1)), label=label)

ax.set_xlabel("electron kinetic energy (eV)")
ax.set_ylabel(spectrum_units())
ax.set_title(f"Level 1: calibrated spectra by GMD bin at {upstream_record['energy']:.3f} eV")
ax.set_xlim(KE_XLIM)
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# Level 2: raw aggregate counts, before GMD normalisation and KE Jacobian.
tof_mask = (upstream_tof_centres >= TOF_RANGE[0]) & (
    upstream_tof_centres <= TOF_RANGE[1]
)
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
for g in range(upstream_agg.n_gmd_bins):
    counts = upstream_agg.D[g] * upstream_agg.n_per_bin[g]
    label = (
        f"[{upstream_agg.gmd_edges[g]:.2g}, "
        f"{upstream_agg.gmd_edges[g + 1]:.2g}) uJ"
    )
    ax.step(upstream_tof_centres[tof_mask], counts[tof_mask], where="mid", lw=0.8, label=label)

ax.set_xlabel("eTOF (100 ps ticks)")
ax.set_ylabel("raw aggregate counts / TOF bin")
ax.set_title(f"Level 2: reconstructed aggregate counts at {upstream_record['energy']:.3f} eV")
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# Level 3: direct shot-resolved H5 histogram versus aggregate reconstruction.
upstream_h5_total = shot_resolved_spectrum(
    upstream_source_h5,
    upstream_tof_edges,
    upstream_agg.gmd_edges,
    trim_start=upstream_trim_start,
    trim_end=upstream_trim_end,
    normalise_by_gmd=False,
)
upstream_agg_total = aggregate_spectrum(upstream_agg, normalise_by_gmd=False)

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
ax.step(
    upstream_tof_centres[tof_mask],
    upstream_h5_total["counts"][tof_mask],
    where="mid",
    lw=0.9,
    label="direct shot-resolved H5",
)
ax.step(
    upstream_tof_centres[tof_mask],
    upstream_agg_total["counts"][tof_mask],
    where="mid",
    lw=0.8,
    alpha=0.7,
    label="aggregate reconstruction",
)
ax.set_xlabel("eTOF (100 ps ticks)")
ax.set_ylabel("raw counts / TOF bin")
ax.set_title(f"Level 3: H5-to-aggregate check at {upstream_record['energy']:.3f} eV")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

difference = upstream_h5_total["counts"] - upstream_agg_total["counts"]
print("H5 shots:", upstream_h5_total["shots"])
print("Aggregate shots:", upstream_agg_total["shots"])
print("Maximum absolute bin difference:", np.nanmax(np.abs(difference)))
print("Total count difference:", np.nansum(difference))

In [ ]:
# Level 4: per-bunch spectral shapes directly from the shot-resolved H5.
def per_bunch_counts_from_h5(
    source_h5, tof_edges, gmd_edges, *, trim_start=0, trim_end=0, chunk_trains=CHUNK_TRAINS
):
    with h5py.File(source_h5, "r") as handle:
        keep = keep_mask_like_compute_aggregates(handle, trim_start, trim_end)
        n_trains, n_bunches = handle["gmd"].shape
        counts = np.zeros((n_bunches, len(tof_edges) - 1), dtype=float)
        for start in range(0, n_trains, chunk_trains):
            stop = min(start + chunk_trains, n_trains)
            row_keep = keep[start:stop]
            if not row_keep.any():
                continue
            gmd = handle["gmd"][start:stop][row_keep]
            tofs = handle["tofs_e"][start:stop][row_keep]
            for bunch in range(n_bunches):
                gmd_bin = np.digitize(gmd[:, bunch], gmd_edges) - 1
                selected = (
                    np.isfinite(gmd[:, bunch])
                    & (gmd_bin >= 0)
                    & (gmd_bin < len(gmd_edges) - 1)
                )
                hits = tofs[selected, bunch].ravel()
                hits = hits[hits > 0]
                counts[bunch] += np.histogram(hits, bins=tof_edges)[0]
    return counts


per_bunch_counts = per_bunch_counts_from_h5(
    upstream_source_h5,
    upstream_tof_edges,
    upstream_agg.gmd_edges,
    trim_start=upstream_trim_start,
    trim_end=upstream_trim_end,
)

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for bunch, counts in enumerate(per_bunch_counts):
    area = counts.sum()
    if area > 0:
        ax.plot(
            upstream_tof_centres[tof_mask],
            counts[tof_mask] / area,
            lw=0.6,
            alpha=0.35,
            label=f"bunch {bunch}" if bunch < 1 else None,
        )
ax.set_xlabel("eTOF (100 ps ticks)")
ax.set_ylabel("area-normalised counts")
ax.set_title(f"Level 4: per-bunch H5 shapes at {upstream_record['energy']:.3f} eV")
ax.grid(alpha=0.2)
plt.show()

In [ ]:
# Upstream QC: check whether the fixed hit-storage depth is reached.
def hit_multiplicity_histogram(
    source_h5, gmd_edges, *, trim_start=0, trim_end=0, chunk_trains=CHUNK_TRAINS
):
    with h5py.File(source_h5, "r") as handle:
        keep = keep_mask_like_compute_aggregates(handle, trim_start, trim_end)
        n_trains = handle["gmd"].shape[0]
        max_slots = int(handle["tofs_e"].shape[-1])
        histogram = np.zeros(max_slots + 1, dtype=np.int64)
        for start in range(0, n_trains, chunk_trains):
            stop = min(start + chunk_trains, n_trains)
            row_keep = keep[start:stop]
            if not row_keep.any():
                continue
            gmd = handle["gmd"][start:stop][row_keep].ravel()
            tofs = handle["tofs_e"][start:stop][row_keep]
            tofs = tofs.reshape(-1, max_slots)
            gmd_bin = np.digitize(gmd, gmd_edges) - 1
            selected = (
                np.isfinite(gmd)
                & (gmd_bin >= 0)
                & (gmd_bin < len(gmd_edges) - 1)
            )
            multiplicity = np.count_nonzero(tofs[selected] > 0, axis=1)
            histogram += np.bincount(multiplicity, minlength=max_slots + 1)
    return histogram


hit_hist = hit_multiplicity_histogram(
    upstream_source_h5,
    upstream_agg.gmd_edges,
    trim_start=upstream_trim_start,
    trim_end=upstream_trim_end,
)
total_shots = hit_hist.sum()
max_slots = len(hit_hist) - 1
print(f"Stored hit slots per shot: {max_slots}")
print(f"Valid shots: {total_shots}")
print(f"Fraction filling every slot: {hit_hist[-1] / total_shots:.6g}")
print("Multiplicity distribution:")
for hits, count in enumerate(hit_hist):
    if count:
        print(f"  {hits:2d} hits: {count:10d}  ({count / total_shots:.6%})")

## Background-subtracted 2D maps

The aggregate and H5 paths below use the same mathematical definition: total eTOF counts divided by summed GMD, followed by division by each calibrated native KE-bin width. The native unequal bins are passed directly to `pcolormesh`; there is no interpolation or smoothing.

Background subtraction is performed only for exact matching photon energies. Signal rows without a measured residual-gas background remain visible in the signal panel but are masked in the background and subtracted panels.

In [ ]:
def matching_background_record(energy, tolerance=1e-6):
    matches = [
        row
        for row in background_records
        if np.isclose(row["energy"], energy, atol=tolerance, rtol=0)
    ]
    if len(matches) > 1:
        raise ValueError(f"Multiple background files match {energy:.3f} eV")
    return matches[0] if matches else None


def make_map_row(record, signal, background=None, source="aggregate"):
    if background is None:
        bg_spectrum = np.full_like(signal["spectrum"], np.nan)
        subtracted = np.full_like(signal["spectrum"], np.nan)
        matched = False
    else:
        bg_spectrum = BACKGROUND_SCALE * background["spectrum"]
        subtracted = signal["spectrum"] - bg_spectrum
        matched = True
    return {
        "energy": record["energy"],
        "family": record["family"],
        "path": record["path"],
        "tof_edges": np.asarray(record["agg"].tof_edges, dtype=float),
        "signal": signal["spectrum"],
        "background": bg_spectrum,
        "subtracted": subtracted,
        "signal_stats": signal,
        "background_stats": background,
        "has_background": matched,
        "source": source,
    }


def build_aggregate_map_rows(dataset=None):
    rows = []
    for record in signal_records:
        if dataset is not None and record["family"] != dataset:
            continue
        signal = aggregate_spectrum(record["agg"])
        bg_record = matching_background_record(record["energy"])
        background = None
        if bg_record is not None:
            if not np.array_equal(record["agg"].tof_edges, bg_record["agg"].tof_edges):
                raise ValueError(f"TOF-edge mismatch at {record['energy']:.3f} eV")
            background = aggregate_spectrum(bg_record["agg"])
        rows.append(make_map_row(record, signal, background, source="aggregate"))
    return rows


def build_h5_map_rows(dataset=None):
    rows = []
    background_cache = {}
    for record in signal_records:
        if dataset is not None and record["family"] != dataset:
            continue
        agg = record["agg"]
        print(f"H5 signal {record['energy']:8.3f} eV  {record['source_h5'].name}")
        signal = shot_resolved_spectrum(
            record["source_h5"],
            agg.tof_edges,
            agg.gmd_edges,
            trim_start=int(agg.metadata.get("trim_start", 0)),
            trim_end=int(agg.metadata.get("trim_end", 0)),
        )
        bg_record = matching_background_record(record["energy"])
        background = None
        if bg_record is not None:
            bg_agg = bg_record["agg"]
            if not np.array_equal(agg.tof_edges, bg_agg.tof_edges):
                raise ValueError(f"TOF-edge mismatch at {record['energy']:.3f} eV")
            cache_key = str(bg_record["source_h5"])
            if cache_key not in background_cache:
                print(f"  H5 background {bg_record['source_h5'].name}")
                background_cache[cache_key] = shot_resolved_spectrum(
                    bg_record["source_h5"],
                    bg_agg.tof_edges,
                    bg_agg.gmd_edges,
                    trim_start=int(bg_agg.metadata.get("trim_start", 0)),
                    trim_end=int(bg_agg.metadata.get("trim_end", 0)),
                )
            background = background_cache[cache_key]
        rows.append(make_map_row(record, signal, background, source="shot-resolved H5"))
    return rows

In [ ]:
def plot_background_maps(
    rows,
    dataset=MAP_DATASET,
    *,
    signal_vmax=SIGNAL_VMAX,
    background_vmax=BACKGROUND_VMAX,
    subtracted_abs_vmax=SUBTRACTED_ABS_VMAX,
):
    selected = sorted(
        [row for row in rows if row["family"] == dataset],
        key=lambda row: (row["energy"], row["path"].name),
    )
    if not selected:
        raise ValueError(f"No rows for dataset {dataset!r}")
    energies = np.array([row["energy"] for row in selected], dtype=float)
    tof_edges = selected[0]["tof_edges"]
    if any(not np.array_equal(tof_edges, row["tof_edges"]) for row in selected[1:]):
        raise ValueError(f"TOF edges differ within dataset {dataset!r}")

    signal_tof = np.vstack([row["signal"] for row in selected])
    background_tof = np.vstack([row["background"] for row in selected])
    subtracted_tof = np.vstack([row["subtracted"] for row in selected])
    ke_edges, signal_ke = tof_window_to_ke_edges_and_data(tof_edges, signal_tof)
    _, background_ke = tof_window_to_ke_edges_and_data(tof_edges, background_tof)
    _, subtracted_ke = tof_window_to_ke_edges_and_data(tof_edges, subtracted_tof)
    energy_edges = half_step_edges(energies)

    signal_vmax = robust_upper(signal_ke) if signal_vmax is None else signal_vmax
    background_vmax = (
        robust_upper(background_ke) if background_vmax is None else background_vmax
    )
    subtracted_abs_vmax = (
        robust_abs(subtracted_ke)
        if subtracted_abs_vmax is None
        else subtracted_abs_vmax
    )
    source = selected[0]["source"]
    normalisation_label = "GMD-normalised" if NORMALISE_BY_GMD else "per-shot"
    jacobian_label = "Jacobian" if APPLY_JACOBIAN else "no Jacobian"
    panels = [
        (signal_ke, "viridis", 0.0, signal_vmax, None, "Signal"),
        (
            background_ke,
            "viridis",
            0.0,
            background_vmax,
            None,
            f"Residual gas x {BACKGROUND_SCALE:g}",
        ),
        (
            subtracted_ke,
            "RdBu_r",
            None,
            None,
            TwoSlopeNorm(
                vmin=-subtracted_abs_vmax,
                vcenter=0.0,
                vmax=subtracted_abs_vmax,
            ),
            "Signal - residual gas",
        ),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True, constrained_layout=True)
    for ax, (data, cmap, vmin, vmax, norm, title) in zip(axes, panels):
        mesh = ax.pcolormesh(
            ke_edges,
            energy_edges,
            data,
            shading="auto",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            norm=norm,
        )
        fig.colorbar(mesh, ax=ax, shrink=0.88, label=spectrum_units())
        ax.set_title(title)
        ax.set_xlabel("electron kinetic energy (eV)")
        ax.set_xlim(KE_XLIM)
        ax.grid(False)
        if INVERT_PHOTON_AXIS:
            ax.set_ylim(np.nanmax(energy_edges), np.nanmin(energy_edges))
    axes[0].set_ylabel("incident photon energy (eV)")
    fig.suptitle(
        f"{dataset} scan from {source}: {normalisation_label}, "
        f"native KE bins, {jacobian_label}"
    )
    plt.show()

    print(f"{dataset} rows ({source}):")
    for row in selected:
        stats = row["signal_stats"]
        bg_label = "matched" if row["has_background"] else "missing"
        print(
            f"  {row['energy']:8.3f} eV  shots={stats['shots']:8d}  "
            f"sumGMD={stats['sum_gmd']:10.5g}  hits={stats['hits']:10.5g}  "
            f"background={bg_label}  {row['path'].name}"
        )
    return fig, axes

In [ ]:
# Fast cross-check using only lightweight aggregate means and bookkeeping.
aggregate_map_rows = build_aggregate_map_rows(dataset=MAP_DATASET)
plot_background_maps(
    aggregate_map_rows,
    dataset=MAP_DATASET,
    signal_vmax=SIGNAL_VMAX,
    background_vmax=BACKGROUND_VMAX,
    subtracted_abs_vmax=SUBTRACTED_ABS_VMAX,
)

In [ ]:
# Primary reconstruction from shot-resolved H5 counts. This is slower but memory-bounded.
h5_map_rows = build_h5_map_rows(dataset=MAP_DATASET)
plot_background_maps(
    h5_map_rows,
    dataset=MAP_DATASET,
    signal_vmax=SIGNAL_VMAX,
    background_vmax=BACKGROUND_VMAX,
    subtracted_abs_vmax=SUBTRACTED_ABS_VMAX,
)

## Selected KE lineouts

This final cell uses the same rows that make the 2D map. It is a direct lineout check of signal, matched residual gas, and their difference without interpolation or smoothing.

In [ ]:
LINEOUT_SOURCE = "h5"  # "h5" or "aggregate"
LINEOUT_DATASET = MAP_DATASET
LINEOUT_ENERGIES = [271.0, 272.0, 273.0, 274.0, 275.0]

lineout_rows = h5_map_rows if LINEOUT_SOURCE == "h5" else aggregate_map_rows
lineout_rows = [row for row in lineout_rows if row["family"] == LINEOUT_DATASET]
fig, axes = plt.subplots(
    len(LINEOUT_ENERGIES),
    1,
    figsize=(10, 2.5 * len(LINEOUT_ENERGIES)),
    sharex=True,
    constrained_layout=True,
)
axes = np.atleast_1d(axes)

for ax, requested_energy in zip(axes, LINEOUT_ENERGIES):
    row = min(lineout_rows, key=lambda item: abs(item["energy"] - requested_energy))
    ke_edges, signal = tof_window_to_ke_edges_and_data(row["tof_edges"], row["signal"])
    _, background = tof_window_to_ke_edges_and_data(row["tof_edges"], row["background"])
    _, subtracted = tof_window_to_ke_edges_and_data(row["tof_edges"], row["subtracted"])
    ke_centres = 0.5 * (ke_edges[:-1] + ke_edges[1:])
    ax.plot(ke_centres, signal, lw=0.9, label="signal")
    if row["has_background"]:
        ax.plot(ke_centres, background, lw=0.9, label="residual gas")
        ax.plot(ke_centres, subtracted, lw=1.0, color="k", label="subtracted")
    else:
        ax.text(0.02, 0.88, "no matched background", transform=ax.transAxes)
    ax.set_ylabel(spectrum_units())
    ax.set_title(
        f"requested {requested_energy:.1f} eV, using {row['energy']:.1f} eV"
    )
    ax.set_xlim(KE_XLIM)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

axes[-1].set_xlabel("electron kinetic energy (eV)")
fig.suptitle(f"{LINEOUT_DATASET} scan lineouts from {LINEOUT_SOURCE}")
plt.show()